# climagrid: forecasting asset environmental stress

This notebook trains a probabilistic forecaster for a grid asset's daily
**environmental stress** and benchmarks it honestly against naive baselines.

> **What this is, and is not.** We forecast a stress feature's future value
> (here, IEEE C57.91 transformer heat-aging stress), not equipment failure.
> climagrid does not predict failures; see the project's Validation Notes.
> A forecast of rising stress is a lead-time aid for scheduling inspections.

It runs unchanged on a laptop or on a free Kaggle / Colab notebook. The data
is tiny, so no GPU is used or needed.

## Setup

Forecasting needs the optional `[ml]` extra (LightGBM, scikit-learn, pyarrow).
On Kaggle, enable **Internet** in the notebook settings so the NASA POWER
fetch works.

```bash
# Released version (once forecasting ships to PyPI):
pip install "climagrid[ml]"
# Unreleased branch (until then):
pip install "climagrid[ml] @ git+https://github.com/TemidireAdesiji/climagrid@feat/forecasting"
```

In [ ]:
import warnings
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import pandas as pd

import climagrid
from climagrid.forecasting import ForecastConfig, evaluate
from climagrid.forecasting.backtest import history_ablation
from climagrid.forecasting.dataset import build_training_panel

warnings.filterwarnings("ignore")
print("climagrid", climagrid.__version__)

## 1. Pick assets and a config

We use a handful of the bundled sample substations to keep the fetch quick.
Point `assets_path` at your own CSV (`asset_id, lat, lon`) to forecast your
own fleet. Increase the asset count and history on Kaggle.

In [ ]:
from pathlib import Path

sample = Path(climagrid.__file__).resolve().parents[2] / "examples" / "data" / "sample_assets.csv"
subset = pd.read_csv(sample, dtype={"asset_id": str}).head(6)
assets_path = "forecast_assets.csv"
subset.to_csv(assets_path, index=False)

config = ForecastConfig(
    targets=["feat_thermal_aging_factor"],
    horizon_days=7,
    lags=[1, 2, 3, 7, 14, 30],
    rolling_windows=[7, 30],
    quantiles=[0.1, 0.5, 0.9],
)

history_start = datetime(2017, 1, 1, tzinfo=timezone.utc)
history_end = datetime(2024, 12, 31, tzinfo=timezone.utc)
subset

## 2. Build the daily training panel

`build_training_panel` fetches hourly climagrid features per asset (streaming
one asset at a time) and aggregates them to one value per day. This is the
slowest step because it calls the NASA POWER API once per asset; the result
can be cached via `ForecastConfig(cache_dir=...)`.

In [ ]:
panel = build_training_panel(assets_path, history_start, history_end, config)
print(panel.shape)
panel.head()

## 3. Backtest honestly against baselines

A forecast is only worth using if it beats persistence (tomorrow = today)
and climatology (day-of-year average) out of sample. `evaluate` runs a
rolling-origin backtest with an embargo gap and reports the skill score
`1 - MSE_model / MSE_baseline` (positive = the model wins).

In [ ]:
scores = evaluate(panel, config, n_splits=3, test_size_days=90)
summary = (
    scores.groupby("horizon_day")[
        ["skill_vs_persistence", "skill_vs_climatology", "interval_coverage"]
    ]
    .mean()
    .round(3)
)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(summary.index, summary["skill_vs_persistence"], marker="o", label="vs persistence")
ax.plot(summary.index, summary["skill_vs_climatology"], marker="s", label="vs climatology")
ax.axhline(0.0, color="grey", lw=0.8, ls="--")
ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("Skill score (1 - MSE_model / MSE_baseline)")
ax.set_title("Forecast skill vs naive baselines")
ax.legend()
fig.tight_layout()
plt.show()

Read this honestly. Stress features are smooth and autocorrelated, so
persistence is a hard baseline at 1 day out and the model may only pull
clearly ahead at longer horizons. `interval_coverage` should sit near 0.8
for a well-calibrated 80% (p10-p90) interval.

## 4. How much history actually helps?

Rather than assume more years is better, measure it. `history_ablation`
reruns the backtest over several history windows so the default length is
chosen on evidence.

In [ ]:
ablation = history_ablation(panel, config, windows_years=[3, 5, 8])
(
    ablation.groupby("history_years")["skill_vs_persistence"].mean().round(3)
    if not ablation.empty
    else "Not enough history in the panel for this ablation."
)

## 5. Forward forecast with prediction intervals

Finally, the deliverable: a forward forecast from each asset's most recent
day, with an 80% interval.

In [ ]:
from climagrid.forecasting.dataset import build_supervised_frame
from climagrid.forecasting.models import LightGBMForecaster

# forecast() is load-and-serve, so train + save a model once, then serve it.
target = config.targets[0]
model = LightGBMForecaster(config).fit(
    build_supervised_frame(panel, target, config), target
)
model.save("thermal_model.joblib")

forecast = climagrid.forecast(assets_path, "thermal_model.joblib", history_end=history_end)
forecast.head(7)

In [ ]:
asset = forecast["asset_id"].iloc[0]
one = forecast[forecast["asset_id"] == asset].sort_values("horizon_day")

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(one["forecast_date"], one["p10"], one["p90"], alpha=0.25, label="p10-p90")
ax.plot(one["forecast_date"], one["p50"], marker="o", label="p50 (median)")
ax.set_title(f"Heat-aging stress forecast: asset {asset}")
ax.set_xlabel("Forecast date")
ax.set_ylabel("feat_thermal_aging_factor")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## Takeaway

We forecast environmental stress, reported skill against honest baselines,
chose the history length by measurement, and produced calibrated intervals.
This is a planning aid for inspection timing, still not a failure prediction.
To go further, combine these forecasts with your own historical failure
records and validate on your own data.